# L3 · The winning 3B in 4 bits: memory, speed, and the confidence for handing a question to the 7B (GPU T4)

Task **L3** (roadmap §7), objective **O4** (the field model "fits in about 6 GB"). Built 23 Nov, while L2 run 4 chose between the two 3Bs.

**Changed on 23 Nov, after the first full run**
- **The overflow:** the first run lost 220 of its 10,675 answers to a half-precision overflow. Each came out as 32 exclamation marks with every score NaN, and was scored wrong. That also left NaN confidences, which stopped the PC notebook. It is L2 step 2b's overflow at the last block of the vision part.
- **The fix:** that layer runs in fp32 and the 3B's vision part under fp16 mixed precision, as in training. Our 7B never overflowed and is only timed here, so it has no fix.
- **The checks:** each answer now also records whether every score was a number. The old check (each chosen token the top score) passed NaN scores. The smoke test adds 12 questions that overflowed.
- **The first run** is kept in `L3-results/full-before-fix/`.
- **Expectation for the rerun:**
  - no answer overflows;
  - memory and speed within 5% of the first run's (the 60 timed questions never overflowed);
  - accuracy at least the first run's on every task, less 0.5 points.

**What it does, in order, on one T4**
1. Times the new 7B (L0 step 2, `AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`) in 4 bits on 60 benchmark questions (10 per task), one at a time, at full size as L0 step 2 asked them: seconds per question and peak GPU memory. The hand-off's time needs it; its answers are L0 step 2's.
2. Loads the winning 3B (base `Qwen/Qwen2.5-VL-3B-Instruct` with its adapter merged in) in fp16 and times it on the same 60.
3. Saves that merged 3B, reloads it in 4 bits (bitsandbytes NF4 with double quantisation, H2b's setting), times it on the same 60, and saves the 4-bit copy to measure its size on disk.
4. The 4-bit 3B answers the benchmark's 10,675 optical questions of the six tasks with L0 step 2's prompts and scorer, greedy, images at the 3B's training size, in batches of 8. With each answer it saves what the answer's confidence needs: each generated token's raw score (logit) and the normaliser (log-sum-exp of all scores) at 10 temperatures, 0.5 to 8. Answers are written as they come, so a stopped run continues.

The hand-off itself is worked out on the PC by `l3-handoff-calibration.ipynb`, from this run's answers and the 7B's saved answers, by the rule below.

**Why NF4 and not AWQ:** the roadmap named AutoAWQ, whose authors stopped maintaining it in 2025. NF4 is the 4-bit setting H2b's memory numbers used and needs nothing new. It saves memory, not time: on a T4 it is expected to be slower than fp16, because it unpacks the weights as it runs. Qwen's own AWQ build is of the base 3B, without our training, so it can't stand in for ours. Path not taken: AWQ of our merged 3B through a maintained tool (llm-compressor), worth it only if GPU speed matters; the field target is a laptop CPU, a separate measurement (roadmap L3 step 6).

**Rule and expectation, fixed before the run (23 Nov)**
- *Confidence of an answer:* the geometric mean of the probabilities of its tokens before the first end token (0 for an empty answer). This is the roadmap's choice (Grace).
- *Calibration (PC):* the questions are split in two by the md5 of their ID: even to the choosing half, odd to the reporting half. The temperature is the one of the 10 with the lowest log loss of confidence against correctness on the choosing half. Reported on the reporting half: the expected calibration error (ECE, 15 equal-width bins) at temperature 1 and at the chosen one, and the risk–coverage curve with its area (RC-AUC). These are the checks TimeSpot asks for.
- *Threshold (PC):* on the choosing half, the threshold that hands the fewest questions to the 7B while the combined accuracy (the 3B's answer at or above it, the 7B's saved answer below it), as the mean of the six tasks' accuracies, stays at least the 7B alone's mean minus 1.0 percentage point. Reported on the reporting half: the share the 3B answers, the combined accuracy, the 7B alone, the 3B alone, and the expected seconds per question (the 3B's time plus the share handed on times the 7B's). Also on E5's clean set (its 186 copied questions dropped).
- *Expectation:* the 4-bit 3B's weights take about 2–3 GB and its peak one question at a time stays under 4 GB, so it meets "about 6 GB" (H2b measured 2.6 GB at peak, without images). The 7B in 4 bits peaks at about 6–7 GB at full size. The 4-bit 3B is slower per question than the fp16 3B and faster than the 4-bit 7B. Four bits cost at most about 2 points against fp16 (L2 run 5) on most tasks. Before scaling the confidence is overconfident, as TimeSpot found for open VLMs: ECE above 0.05 at temperature 1, and a temperature above 1 chosen. No share for the hand-off was predicted.

**How to run on Kaggle** (account 3, `redwannewaz`)
1. Settings: GPU **T4 x2** (one is used), internet ON. Secret `HF_TOKEN` (AbrarAlam: the adapters and the 7B are in private repos), or attach `redwannewaz/l2-hf-write-token` when the run is started from the PC.
2. Inputs: `redwannewaz/disasterm3-bench-mirror-v2`.
3. In cell 2, `WINNER` is L2 run 4's winner, from its `selection.json`: `"c"` since 23 Nov.
4. Run All once with `SMOKE_TEST = True` (5 questions per task, 2 per task timed). Then `False` and Save & Run All. About 4–7 h (estimated). It stops cleanly 11 h after it starts; to continue, attach that version's output and run again.
5. Outputs in `/kaggle/working/L3/`: `results.csv`, `per_question.csv`, `speed.csv`, `speed.json`, `answers.jsonl`, `README.md`, `config.json`. Save them in `Champion_Task/L3-results/full/L3/` (a smoke test's in `L3-results/smoke/L3/`), then run `l3-handoff-calibration.ipynb` on the PC.

In [ ]:
# ── Cell 1: Environment: Kaggle's torch and transformers, current peft and bitsandbytes, and no torchao ──
# L0's merge (26 Oct) stopped on "Found an incompatible version of torchao": peft checks torchao when it puts an adapter
# on an fp16 model, as this notebook does. Nothing here uses torchao.
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"
import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft", "bitsandbytes")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 1, "Pick GPU T4 x2 (one is used)"

In [ ]:
# ── Cell 2: Configuration, the benchmark's prompts and scorer, the models, and each answer's confidence ──
import ast, gc, glob, json, math, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig

T_START = time.time()
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_REPO = "AbrarAlam/disasterm3-qwen25vl3b-l2"
SEVEN_B = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"   # the new 7B (L0 step 2): timed here; its answers are L0 step 2's
WINNER = "c"                   # L2 run 4's winner (its selection.json, 23 Nov): (c), the options rerun
BATCH, MAX_NEW_TOKENS = 8, 32
SMOKE_TEST = False             # True = 5 questions per task
TIME_LIMIT_HOURS = 11.0        # counted from this cell
SPEED_PER_TASK = 2 if SMOKE_TEST else 10                 # questions per task timed one at a time
TEMPS = (0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0)   # the temperatures whose normalisers are saved, for calibration
OUT_DIR = Path("/kaggle/working/L3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH, SPEED_PATH = OUT_DIR / "answers.jsonl", OUT_DIR / "speed.json"
MERGED, NF4_COPY = Path("/tmp/l3_merged_fp16"), Path("/tmp/l3_nf4")    # outside the output: 7.5 GB and 2-3 GB
CUDA = torch.cuda.is_available()
NF4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)                 # H2b's 4-bit setting
EXPECTED_QUESTIONS = {"DTR": 353, "BBR": 2006, "BDC": 3583, "DRE": 1868, "DSR": 2007, "ORR": 858}
REF_NEW_7B_FULL = {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62}   # L0 step 2
REF_BASE_7B = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}              # DisasterM3 Table 2
assert WINNER in ("b", "c"), "set WINNER to L2 run 4's winner (its selection.json)"

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}

def load_3b(which):
    """The base 3B in fp16 on one T4; for 'b' or 'c', its finished adapter merged in."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": 0} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return model.eval()

# ── The overflow fix (23 Nov): L2 step 2b's layer in fp32, inside a vision part run under fp16 autocast, as in training ──
# L3's first full run lost 220 of its 10,675 answers to a half-precision overflow at this layer: each came out as 32
# exclamation marks, the token picked when every score is NaN (run 5's first run lost 327). Applied to the 3B only: our
# 7B never overflowed (L0 step 2), and its answers here are only timed.
FP32_LAYER = "visual.blocks.31.mlp.down_proj"
DEVICE = "cuda" if CUDA else "cpu"
OVERFLOWED_BEFORE = ["disaster_type_331", "disaster_type_335", "bearing_body_851", "bearing_body_1645",
                     "building_damage_counting_1059", "building_damage_counting_1908", "road_damage_counting_909",
                     "road_damage_counting_1582", "landuse_1454", "landuse_1465", "relational_reasoning_qa_877",
                     "relational_reasoning_qa_878"]   # overflowed in L3's first full run (2 per task): the smoke test adds them
OVERFLOWED = re.compile(r"!+")

def fp32_islands(model, names):
    """L2 step 2's function: these modules run in fp32 with autocast off (their inputs, their fp16 weights and their
    4-bit layers' compute)."""
    mods = dict(model.named_modules())
    for n in names:
        m = mods[n]

        def forward(*args, __orig=m.forward, **kwargs):
            with torch.autocast(DEVICE, enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                kwargs = {k: v.float() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in kwargs.items()}
                return __orig(*args, **kwargs)

        m.forward = forward
        for sub in m.modules():
            if hasattr(sub, "compute_dtype"):
                sub.compute_dtype = torch.float32
            for p in sub.parameters(recurse=False):
                if p.is_floating_point() and p.dtype != torch.float32:
                    p.data = p.data.float()

def with_fix(model):
    """Step 2b's layer in fp32 inside a vision part run under fp16 autocast. As in training, no 4-bit layer switches its
    compute dtype to its input's, so only step 2b's layer computes in fp32."""
    for mod in model.modules():
        if hasattr(mod, "compute_type_is_set"):
            mod.compute_type_is_set = True
    islands = [n for n, _ in model.named_modules() if n.endswith(FP32_LAYER)]
    towers = [m for n, m in model.named_modules() if n.endswith("visual") and hasattr(m, "blocks")]
    assert len(islands) == 1 and len(towers) == 1, f"{len(islands)} of step 2b's layer, {len(towers)} vision parts"
    fp32_islands(model, islands)
    orig = towers[0].forward

    def forward(*args, **kwargs):
        with torch.autocast(DEVICE, dtype=torch.float16):
            return orig(*args, **kwargs)

    towers[0].forward = forward
    return model

def load_nf4(source):
    """A Qwen2.5-VL in 4 bits (NF4) on one T4, quantised as it loads."""
    from transformers import Qwen2_5_VLForConditionalGeneration
    return Qwen2_5_VLForConditionalGeneration.from_pretrained(str(source), quantization_config=NF4, dtype=torch.float16,
                                                              device_map={"": 0}, attn_implementation="sdpa",
                                                              token=hf_token()).eval()

def cleared():
    gc.collect()
    torch.cuda.empty_cache()

def folder_gb(path):
    return round(sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9, 2)

EMPTY = {"z": [], "lse": [], "greedy_ok": True, "finite": True}

def token_stats(logits, gen, end_ids):
    """Per answer, for the tokens before its first end token: each chosen token's raw score z, and log-sum-exp(scores / T)
    for each T in TEMPS. Then log p_T(token) = z / T - lse_T, without keeping the 151,936 scores per token. greedy_ok
    says each chosen token was the top score, i.e. no setting changed the scores before the choice. finite says every
    score was a number: an overflow makes them all NaN (23 Nov), which argmax alone doesn't notice."""
    z = torch.stack(logits, 1).float()                                   # (answers, steps, vocabulary)
    chosen = gen[:, :z.shape[1]]
    zc = z.gather(2, chosen.unsqueeze(-1)).squeeze(-1)
    lse = torch.stack([torch.logsumexp(z / t, -1) for t in TEMPS], -1)   # (answers, steps, temperatures)
    ok = torch.isfinite(z).all(-1)                                        # (answers, steps)
    top = (z.argmax(-1) == chosen) & ok
    stats = []
    for i, ids in enumerate(chosen.tolist()):
        n = next((k for k, t in enumerate(ids) if t in end_ids), len(ids))
        stats.append({"z": [round(v, 4) for v in zc[i, :n].tolist()],
                      "lse": [[round(v, 4) for v in row] for row in lse[i, :n].tolist()],
                      "greedy_ok": bool(top[i, :n].all()), "finite": bool(ok[i, :max(n, 1)].all())})
    return stats

def confidence(stat, t=1.0):
    """The geometric mean of the answer's token probabilities at temperature t (0 for an empty answer)."""
    if not stat["z"]:
        return 0.0
    k = TEMPS.index(t)
    return math.exp(sum(z / t - l[k] for z, l in zip(stat["z"], stat["lse"])) / len(stat["z"]))

# The arithmetic on a made-up answer: tokens 2 and 1, then the end token 0, over a vocabulary of 4
_z = (torch.tensor([[0., 1., 3., 0.]]), torch.tensor([[0., 2., 1., 0.]]), torch.tensor([[5., 0., 0., 0.]]))
_s = token_stats(_z, torch.tensor([[2, 1, 0]]), {0})[0]
_p = torch.softmax(_z[0][0], 0)[2].item() * torch.softmax(_z[1][0], 0)[1].item()
assert len(_s["z"]) == 2 and _s["greedy_ok"] and _s["finite"] and abs(confidence(_s) - math.sqrt(_p)) < 1e-3
_nan = token_stats((torch.full((1, 4), float("nan")),), torch.tensor([[0]]), {5})[0]   # an overflowed answer
assert not _nan["greedy_ok"] and not _nan["finite"]
assert abs(confidence(_s, 2.0) - math.sqrt(torch.softmax(_z[0][0] / 2, 0)[2].item() * torch.softmax(_z[1][0] / 2, 0)[1].item())) < 1e-3
assert confidence(EMPTY) == 0.0

@torch.inference_mode()
def answer_batch(model, proc, items, size=sized):
    """items: (prompt text, image paths) -> (greedy answers, their token statistics). size=None leaves the images as they
    are (the 7B at full size, as L0 step 2 asked it)."""
    texts, images = [], []
    for text, paths in items:
        imgs = [Image.open(p).convert("RGB") for p in paths]
        imgs = [size(im) for im in imgs] if size else imgs
        texts.append(proc.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                              add_generation_prompt=True))
        images += imgs
    inputs = proc(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0, output_logits=True, return_dict_in_generate=True)
    gen = out.sequences[:, inputs["input_ids"].shape[1]:]
    return ([s.strip() for s in proc.batch_decode(gen, skip_special_tokens=True)], token_stats(out.logits, gen, END_IDS))

def answer_safe(model, proc, items, size=sized):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    confidence 0, as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, proc, items, size)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""], [EMPTY]
        half = len(items) // 2
        a1, s1 = answer_safe(model, proc, items[:half], size)
        a2, s2 = answer_safe(model, proc, items[half:], size)
        return a1 + a2, s1 + s2

def measure(model, proc, rows, size=sized):
    """One question at a time, as in the field: the GPU memory the weights hold, the peak while answering, seconds per
    question, and generated tokens per second (the end token counted). One untimed question first warms the GPU up."""
    weights_gb = torch.cuda.memory_allocated() / 1e9
    answer_batch(model, proc, rows[:1], size)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    secs, tokens, answers = [], [], []
    for row in rows:
        t = time.perf_counter()
        a, s = answer_batch(model, proc, [row], size)
        torch.cuda.synchronize()
        secs.append(time.perf_counter() - t)
        tokens.append(len(s[0]["z"]) + 1)
        answers += a
    return {"questions": len(rows), "weights_gpu_gb": round(weights_gb, 2),
            "footprint_gb": round(model.get_memory_footprint() / 1e9, 2),
            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "s_per_question_mean": round(float(np.mean(secs)), 3), "s_per_question_median": round(float(np.median(secs)), 3),
            "tokens_per_s": round(sum(tokens) / sum(secs), 2), "answers": answers}

BENCH_JSON = one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2")
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.padding_side = "left"
_tok = processor.tokenizer
END_IDS = {i for i in (_tok.convert_tokens_to_ids("<|im_end|>"), _tok.eos_token_id, _tok.pad_token_id) if i is not None}
print(f"✓ L3{'  [SMOKE TEST]' if SMOKE_TEST else ''}: the winner ({WINNER}) in 4 bits; end tokens {sorted(END_IDS)}")

In [ ]:
# ── Cell 3: The benchmark's optical questions of the six tasks (as L0 step 2 asked them), and the 60 timed ──
with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bench_index = {}                # every benchmark image by file name (L0 step 2's rule: mirrors nest them differently)
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

def sensor(e):
    """H1 V2's rule for the after-image's sensor: most entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

rows = []
for code, subset, bench_task, _ in TASKS:      # ids as in H1 V2, L0 step 2 and E5: the index among the task's entries
    entries = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == bench_task)]
    entries = [(qid, e) for qid, e in entries if sensor(e) == "Optical"]
    assert len(entries) == EXPECTED_QUESTIONS[code], f"{code}: {len(entries)} optical questions"
    for qid, e in entries:
        text, keys = prompt_text(e, subset)
        row = {"id": qid, "task": code, "text": text, "images": [bench_index.get(name_of(e[k])) for k in keys], "truth": e["ground_truth_option"]}
        if code in COUNTING:
            values = option_values(e)
            right = sorted(letters(e["ground_truth_option"]))[0]
            row.update(values=values, truth_value=values["ABCDE".index(right)], rule_correct=no_image_rule(values) == right)
        rows.append(row)
Q = pd.DataFrame(rows)
Q["image_missing"] = Q["images"].map(lambda ps: any(p is None for p in ps))
assert Q["image_missing"].sum() <= 0.01 * len(Q), f"{Q['image_missing'].sum()} questions have an image missing: attach the bench mirror"
assert set(OVERFLOWED_BEFORE) <= set(Q["id"]), "a question that overflowed before is missing"
print(f"✓ {len(Q):,} optical questions { {k: int(v) for k, v in Q['task'].value_counts().items()} }; "
      f"{int(Q['image_missing'].sum())} with an image missing "
      "(counted wrong)")
if SMOKE_TEST:                  # 5 per task, and (23 Nov) the 12 that overflowed in the first full run
    Q = Q[Q.index.isin(Q.groupby("task").head(5).index) | Q["id"].isin(OVERFLOWED_BEFORE)].reset_index(drop=True)
SPEED = Q[~Q["image_missing"]].groupby("task").head(SPEED_PER_TASK)    # the first questions of each task, the same every run
SPEED_ROWS = list(zip(SPEED["text"], SPEED["images"]))
print(f"✓ {len(SPEED_ROWS)} questions timed one at a time")

In [ ]:
# ── Cell 4: Memory and speed (the 7B and the 3B in 4 bits, the 3B in fp16), then the benchmark in 4 bits ──
from tqdm.auto import tqdm

for name, path in (("speed.json", SPEED_PATH), ("answers.jsonl", ANSWERS_PATH)):    # a stopped run's output, if attached
    previous = sorted(glob.glob(f"/kaggle/input/**/L3/{name}", recursive=True))
    if previous and not path.exists():
        shutil.copy(previous[0], path)
speed = json.loads(SPEED_PATH.read_text(encoding="utf-8")) if SPEED_PATH.exists() else {}

def save_speed():
    SPEED_PATH.write_text(json.dumps(speed, indent=1), encoding="utf-8")

def show(key):
    print(f"✓ {key}: " + ", ".join(f"{k} {v}" for k, v in speed[key].items() if k != "answers")
          + f"; /tmp has {shutil.disk_usage('/tmp').free / 1e9:.0f} GB free")

# 1. The new 7B in 4 bits, at full size: the hand-off's time and memory when it hands a question on
if "7b_nf4" not in speed:
    proc_7b = AutoProcessor.from_pretrained(SEVEN_B, token=hf_token())
    proc_7b.tokenizer.padding_side = "left"
    model = load_nf4(SEVEN_B)
    speed["7b_nf4"] = measure(model, proc_7b, SPEED_ROWS, size=None)
    model = None
    cleared()
    shutil.rmtree(Path(HF_HUB_CACHE) / ("models--" + SEVEN_B.replace("/", "--")), ignore_errors=True)   # 16 GB of disk back
    save_speed()
show("7b_nf4")

# 2. The winning 3B in fp16 (base and adapter merged): saved as it is to be loaded in 4 bits, then timed with the fix
model = load_3b(WINNER)
model.save_pretrained(MERGED)
with_fix(model)
if "3b_fp16" not in speed:
    speed["3b_fp16"] = measure(model, processor, SPEED_ROWS)
speed["3b_fp16"]["disk_gb"] = folder_gb(MERGED)
save_speed()
show("3b_fp16")
model = None
cleared()

# 3. The same merged 3B in 4 bits: its 4-bit copy's size on disk, then timed with the fix
model = load_nf4(MERGED)
if "3b_nf4" not in speed:
    try:
        model.save_pretrained(NF4_COPY)
        nf4_disk_gb = folder_gb(NF4_COPY)
    except Exception as e:            # a side number: the run goes on without it
        nf4_disk_gb = None
        print(f"⚠ the 4-bit copy wasn't saved: {e}")
with_fix(model)
if "3b_nf4" not in speed:
    speed["3b_nf4"] = measure(model, processor, SPEED_ROWS)
    speed["3b_nf4"]["disk_gb"] = nf4_disk_gb
    save_speed()
show("3b_nf4")

# 4. The benchmark in 4 bits, each answer with its token statistics (resumable)
TAG = f"{WINNER}-nf4-fix"                 # 23 Nov: answers made with the overflow fix only, never the first run's
done = {}
if ANSWERS_PATH.exists():
    with open(ANSWERS_PATH, encoding="utf-8") as f:
        done = {r["id"]: r for r in map(json.loads, f) if r.get("model") == TAG}
todo = Q[~Q["id"].isin(list(done)) & ~Q["image_missing"]]
print(f"{len(done):,} already answered; {len(todo):,} to do")
deadline, t0, stopped = T_START + TIME_LIMIT_HOURS * 3600, time.time(), False
with open(ANSWERS_PATH, "a", encoding="utf-8") as fh:
    for start in tqdm(range(0, len(todo), BATCH), desc=f"3B ({TAG})"):
        if time.time() > deadline:
            stopped = True
            break
        batch = todo.iloc[start:start + BATCH]
        answers, stats = answer_safe(model, processor, list(zip(batch["text"], batch["images"])))
        for qid, answer, stat in zip(batch["id"], answers, stats):
            done[qid] = {"id": qid, "model": TAG, "answer": answer, **stat}
            fh.write(json.dumps(done[qid]) + "\n")
        fh.flush()
RUN_HOURS = (time.time() - t0) / 3600
print(f"{'Time limit reached: stopped cleanly' if stopped else 'All answered'} after {RUN_HOURS:.2f} h")

In [ ]:
# ── Cell 5: Scores in 4 bits against the references, memory and speed, and the output files ──
Q["answer"] = Q["id"].map(lambda i: done[i]["answer"] if i in done else None)
Q["confidence"] = Q["id"].map(lambda i: round(confidence(done[i]), 5) if i in done else None)
Q["n_tokens"] = Q["id"].map(lambda i: len(done[i]["z"]) if i in done else None)
Q["correct"] = [a is not None and letters(a) == letters(t) for a, t in zip(Q["answer"], Q["truth"])]
complete = bool((Q["answer"].notna() | Q["image_missing"]).all())
greedy_ok = float(np.mean([r["greedy_ok"] for r in done.values()])) if done else float("nan")
overflowed = int(sum(bool(OVERFLOWED.fullmatch(r["answer"].strip())) for r in done.values()))
nonfinite = int(sum(not r.get("finite", True) for r in done.values()))
print(f"Overflowed answers (only exclamation marks): {overflowed}; answers with a score that isn't a number: {nonfinite}")
if SMOKE_TEST:
    print("The 12 that overflowed in the first full run, now:", [done[i]["answer"] if i in done else None for i in OVERFLOWED_BEFORE])
table = []
for code, _, _, _ in TASKS:
    g = Q[Q["task"] == code]
    table.append({"task": code, "questions": len(g), "answered": int(g["answer"].notna().sum()),
                  f"3B_{WINNER}_4bit_%": round(100 * g["correct"].mean(), 2) if len(g) else float("nan"),
                  "new_7B_full_%": REF_NEW_7B_FULL[code], "base_7B_paper_%": REF_BASE_7B[code],
                  "mean_confidence": round(float(g["confidence"].mean()), 3) if g["confidence"].notna().any() else None})
results = pd.DataFrame(table)
bdc = Q[Q["task"] == "BDC"].copy()
bdc["value"] = [chosen_value(a, v) if a is not None else None for a, v in zip(bdc["answer"], bdc["values"])]
err = (bdc["value"].astype(float) - bdc["truth_value"]).abs() / bdc["truth_value"].clip(lower=1)
counting = {"accuracy_%": round(100 * float(bdc["correct"].mean()), 2),
            "median_count_error_%": round(100 * float(err.median()), 1) if err.notna().any() else None,
            "answered_with_one_option_%": round(100 * float(bdc["value"].notna().mean()), 1),
            "no_image_rule_%": round(100 * float(bdc["rule_correct"].mean()), 2)}
LABELS = {"7b_nf4": "new 7B, 4 bits, full size", "3b_fp16": f"3B ({WINNER}), fp16", "3b_nf4": f"3B ({WINNER}), 4 bits"}
speed_table = pd.DataFrame([{"model": LABELS[k], **{a: b for a, b in v.items() if a != "answers"}} for k, v in speed.items()])
same = lambda a, b: float(np.mean([letters(x) == letters(y) for x, y in zip(a, b)]))
batch_nf4 = [done[i]["answer"] if i in done else "" for i in SPEED["id"]]
agreement = {"3B fp16 vs 4 bits, one at a time": round(same(speed["3b_fp16"]["answers"], speed["3b_nf4"]["answers"]), 3),
             "3B 4 bits, one at a time vs in batches of 8": round(same(speed["3b_nf4"]["answers"], batch_nf4), 3)}
print(results.to_string(index=False))
print(f"\nBuilding counting: {counting}\n")
print(speed_table.to_string(index=False))
print(f"\nSame answers on the timed questions: {agreement}; chosen tokens that were the top score: {greedy_ok:.4f}")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

Q.drop(columns=["text"]).to_csv(OUT_DIR / "per_question.csv", index=False)
results.to_csv(OUT_DIR / "results.csv", index=False)
speed_table.to_csv(OUT_DIR / "speed.csv", index=False)
config = {"task": "L3: the winning 3B in 4 bits, memory, speed and confidence", "smoke_test": SMOKE_TEST, "winner": WINNER,
          "base_model": BASE_MODEL, "adapter": f"{ADAPTER_REPO}/{WINNER}/final", "seven_b": SEVEN_B,
          "quantisation": "bitsandbytes NF4, double quantisation, fp16 compute", "max_pixels": MAX_PIXELS, "decoding": "greedy",
          "batch": BATCH, "timed_questions": len(SPEED_ROWS), "temperatures": TEMPS, "complete": complete,
          "fix": f"{FP32_LAYER} in fp32; the vision part under fp16 autocast (23 Nov), on the 3B only",
          "overflowed_answers": overflowed, "nonfinite_answers": nonfinite,
          "greedy_ok_share": greedy_ok, "agreement": agreement, "hours_answering": round(RUN_HOURS, 2),
          "hours_total": round((time.time() - T_START) / 3600, 2), "building_counting": counting, "versions": VERSIONS}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    f"# L3: the winning 3B, option ({WINNER}), in 4 bits" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"`{BASE_MODEL}` with the adapter `{ADAPTER_REPO}/{WINNER}/final` merged in, then loaded in 4 bits (bitsandbytes NF4), "
    f"on the benchmark's optical questions of the six tasks, with L0 step 2's prompts and scorer, greedy, images at most "
    f"{MAX_PIXELS:,} pixels, in batches of {BATCH}.", "",
    "## Memory and speed on one T4, one question at a time", "", as_markdown(speed_table), "",
    f"Timed on {len(SPEED_ROWS)} questions ({SPEED_PER_TASK} per task). `weights_gpu_gb` is the GPU memory after loading, "
    "`peak_gpu_gb` the most while answering, `tokens_per_s` the generated tokens (the end token counted) over the time per "
    "question, which is mostly reading the images. The 7B reads the images at full size, as L0 step 2 asked it.", "",
    f"Same answers on the timed questions: {agreement}.", "",
    "## Accuracy in 4 bits (%)", "", as_markdown(results), "",
    "The new 7B's numbers are at full size (L0 step 2); the base 7B's are the DisasterM3 paper's (Table 2). The 3B in fp16 "
    "is L2 run 5 (`l2-step4-winner-benchmark`).", "",
    "## Building counting, beside accuracy", "",
    f"- Accuracy {counting['accuracy_%']}%; median count error {counting['median_count_error_%']}%, on the "
    f"{counting['answered_with_one_option_%']}% answered with one option.",
    f"- The rule that never sees the image: {counting['no_image_rule_%']}%.", "",
    "## Confidence", "",
    f"Each answer's token scores are in `answers.jsonl`; `per_question.csv` has its confidence at temperature 1 (the geometric "
    f"mean of its token probabilities). {100 * greedy_ok:.2f}% of answers had every chosen token at the top score. "
    "Calibration and the hand-off to the 7B: `l3-handoff-calibration.ipynb` on the PC.", "",
    "## The overflow fix (23 Nov)", "",
    f"`{FP32_LAYER}` runs in fp32 and the 3B's vision part under fp16 autocast, as in training (L2 step 2b). Overflowed "
    f"answers: {overflowed}; answers with a score that isn't a number: {nonfinite} (the first full run, without the fix: 220).", "",
    "## Files", "", "- `results.csv`, `per_question.csv`, `speed.csv`, `speed.json`, `answers.jsonl`, `config.json`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT_DIR}: results.csv, per_question.csv, speed.csv, speed.json, answers.jsonl, README.md, config.json"
      + ("" if complete else " (not complete: attach this output and run again)"))